In [5]:
from google.colab import files

uploaded = files.upload()

Saving puzzles.txt to puzzles (1).txt


In [6]:
with open("puzzles.txt", "r") as file:
    text = file.read()

puzzles = text.split()

print("Number of puzzles:", len(puzzles))

for i, puzzle in enumerate(puzzles, 1):
    print("Puzzle", i)
    print(puzzle)
    print("Length:", len(puzzle))
    print()

Number of puzzles: 3
Puzzle 1
003020600900305001001806400008102900700000008006708200002609500800203009005010300
Length: 81

Puzzle 2
000000000000003085001020000000507000004000100090000000500000073002010000000040009
Length: 81

Puzzle 3
800000000003600000070090200050007000000045700000100030001000068008500010090000400
Length: 81



In [7]:
class Sudoku:

    def __init__(self, puzzle):

        self.puzzle = puzzle
        self.variables = list(range(81))
        self.neighbors = self.make_neighbors()


        self.domains = {}
        for i in range(81):
            if puzzle[i] == '0':
                self.domains[i] = set(range(1, 10))
            else:
                self.domains[i] = {int(puzzle[i])}


    def make_neighbors(self):
        neighbors = {}
        for cell in range(81):
            row = cell // 9
            col = cell % 9

            neighbors[cell] = set()
            for c in range(9):
                neighbors[cell].add(row * 9 + c)
            for r in range(9):
                neighbors[cell].add(r * 9 + col)
            start_row = (row // 3) * 3
            start_col = (col // 3) * 3
            for r in range(start_row, start_row + 3):
                for c in range(start_col, start_col + 3):
                    neighbors[cell].add(r * 9 + c)


            neighbors[cell].discard(cell)

        return neighbors

In [8]:
from collections import deque


def revise(sudoku, domains, xi, xj):
    changed = False
    for x in list(domains[xi]):
        possible = False
        for y in domains[xj]:
            if x != y:
                possible = True
                break
        if not possible:
            domains[xi].remove(x)
            changed = True
    return changed


def AC3(sudoku, domains):
    queue = deque()
    for xi in range(81):
        for xj in sudoku.neighbors[xi]:
            queue.append((xi, xj))

    while queue:
        xi, xj = queue.popleft()
        if revise(sudoku, domains, xi, xj):
            if len(domains[xi]) == 0:
                return False
            for xk in sudoku.neighbors[xi]:
                if xk != xj:
                    queue.append((xk, xi))

    return True

In [9]:
puzzle = puzzles[0]

sudoku = Sudoku(puzzle)

print("Before AC-3:")
print(sudoku.domains[1])

AC3(sudoku, sudoku.domains)

print("After AC-3:")
print(sudoku.domains[1])

Before AC-3:
{1, 2, 3, 4, 5, 6, 7, 8, 9}
After AC-3:
{8}


In [10]:
def MRV(sudoku, domains):

    unassigned = []
    for cell in range(81):
        if len(domains[cell]) > 1:
            unassigned.append(cell)

    if len(unassigned) == 0:
        return None

    return min(unassigned,key=lambda cell: len(domains[cell]))

In [11]:
def backtracking(sudoku, domains):
    cell = MRV(sudoku, domains)
    if cell is None:
        return domains

    for value in sorted(domains[cell]):
        new_domains = {}
        for i in range(81):
            new_domains[i] = domains[i].copy()

        new_domains[cell] = {value}
        if AC3(sudoku, new_domains):
            result = backtracking(sudoku,new_domains)

            if result is not None:
                return result



    return None

In [12]:
def get_solution(domains):

    answer = ""

    for i in range(81):

        answer += str(next(iter(domains[i])))

    return answer

In [13]:
solutions = []

for number, puzzle in enumerate(puzzles, 1):
    print("Solving puzzle", number)
    sudoku = Sudoku(puzzle)

    if not AC3(sudoku, sudoku.domains):
        print("No solution")
        continue

    answer = backtracking(sudoku,sudoku.domains )

    if answer is not None:
        solution = get_solution(answer)
        solutions.append(solution)
        print("Solved:")
        print(solution)

    else:
        print("No solution")
    print()

Solving puzzle 1
Solved:
483921657967345821251876493548132976729564138136798245372689514814253769695417382

Solving puzzle 2
Solved:
987654321246173985351928746128537694634892157795461832519286473472319568863745219

Solving puzzle 3
Solved:
812753649943682175675491283154237896369845721287169534521974368438526917796318452

